# LeetCode 2146 — K Highest Ranked Items Within a Price Range bằng Python

Link đề: https://leetcode.com/problems/k-highest-ranked-items-within-a-price-range/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Sorting, Heap (Priority Queue), Matrix

Skeleton LeetCode:

```python
class Solution:
    def highestRankedKItems(self, grid: List[List[int]], pricing: List[int], start: List[int], k: int) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Dùng **BFS trên lưới** để tính khoảng cách ngắn nhất (mỗi bước trọng số 1) từ một ô xuất phát, có tường chặn.
2. Mã hóa tiêu chí xếp hạng nhiều tầng bằng **tuple** `(khoảng cách, giá, hàng, cột)` — Python so sánh tuple theo thứ tự từ điển.
3. Chọn `k` phần tử tốt nhất bằng: **sắp xếp toàn bộ**, `heapq.nsmallest`, **heap kích thước k**, và **BFS theo tầng + dừng sớm**.
4. Tránh bẫy: quên ô xuất phát (khoảng cách 0), đánh dấu thăm muộn (đẩy trùng), sửa trực tiếp `grid`, nhầm thứ tự tiêu chí.
5. Kiểm chứng với một bộ tính khoảng cách độc lập (relax lặp kiểu Bellman–Ford) trên lưới nhỏ ngẫu nhiên.

## 2. Đề bài & ví dụ

Lưới `grid` `m × n` là bản đồ cửa hàng:
- `0`: tường, không đi qua được.
- `1`: ô trống, đi lại tự do.
- số khác (≥ 2): **giá** của món hàng ở ô đó; cũng đi lại tự do.

Mỗi bước đi sang ô kề cạnh tốn 1. Cho `pricing = [low, high]`, `start = [row, col]` và `k`. Chỉ quan tâm các món có giá trong `[low, high]` và **đi tới được** từ `start`.

**Hạng** của món hàng xét lần lượt (tiêu chí đầu tiên khác nhau quyết định):
1. **Khoảng cách** ngắn nhất từ `start` (gần hơn ⇒ hạng cao hơn);
2. **Giá** (rẻ hơn ⇒ cao hơn);
3. **Hàng** (nhỏ hơn ⇒ cao hơn);
4. **Cột** (nhỏ hơn ⇒ cao hơn).

Trả về vị trí của `k` món hạng cao nhất (theo thứ tự hạng giảm dần); nếu ít hơn `k` món thì trả tất cả.

**Ví dụ 1:** `grid = [[1,2,0,1],[1,3,0,1],[0,2,5,1]], pricing = [2,5], start = [0,0], k = 3` → `[[0,1],[1,1],[2,1]]`.

**Ví dụ 2:** `grid = [[1,2,0,1],[1,3,3,1],[0,2,5,1]], pricing = [2,3], start = [2,3], k = 2` → `[[2,1],[1,2]]` (cùng khoảng cách 2, giá 2 < 3).

**Ví dụ 3:** `grid = [[1,1,1],[0,0,1],[2,3,4]], pricing = [2,3], start = [0,0], k = 3` → `[[2,1],[2,0]]` (chỉ có 2 món).

**Ràng buộc:** `1 ≤ m·n ≤ 10⁵`, `0 ≤ grid[i][j] ≤ 10⁵`, `2 ≤ low ≤ high ≤ 10⁵`, `grid[row][col] > 0`, `1 ≤ k ≤ m·n`.

In [1]:
from typing import List
from collections import deque
import heapq, random, time, copy

examples = [
    ([[1, 2, 0, 1], [1, 3, 0, 1], [0, 2, 5, 1]], [2, 5], [0, 0], 3, [[0, 1], [1, 1], [2, 1]]),
    ([[1, 2, 0, 1], [1, 3, 3, 1], [0, 2, 5, 1]], [2, 3], [2, 3], 2, [[2, 1], [1, 2]]),
    ([[1, 1, 1], [0, 0, 1], [2, 3, 4]], [2, 3], [0, 0], 3, [[2, 1], [2, 0]]),
]
def show(grid, start):
    for r, row in enumerate(grid):
        print("   " + " ".join(("S" if [r, c] == start else ("#" if v == 0 else str(v))).rjust(2) for c, v in enumerate(row)))
for g, p, s, k, exp in examples:
    print(f"pricing={p}, start={s}, k={k} -> kỳ vọng {exp}"); show(g, s)

pricing=[2, 5], start=[0, 0], k=3 -> kỳ vọng [[0, 1], [1, 1], [2, 1]]
    S  2  #  1
    1  3  #  1
    #  2  5  1
pricing=[2, 3], start=[2, 3], k=2 -> kỳ vọng [[2, 1], [1, 2]]
    1  2  #  1
    1  3  3  1
    #  2  5  S
pricing=[2, 3], start=[0, 0], k=3 -> kỳ vọng [[2, 1], [2, 0]]
    S  1  1
    #  #  1
    2  3  4


## 3. Kiến thức nền

### 3.1 BFS trên lưới

Hàng đợi `deque`, đánh dấu `visited` **ngay khi đẩy vào hàng đợi** (không phải khi lấy ra) để mỗi ô vào hàng đợi đúng một lần. Lần đầu tiên một ô được chạm tới chính là khoảng cách ngắn nhất (vì BFS đi theo tầng).

In [2]:
def bfs_dist(grid, start):
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]
    r0, c0 = start; dist[r0][c0] = 0
    q = deque([(r0, c0)])
    while q:
        r, c = q.popleft()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != 0 and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

for row in bfs_dist(examples[0][0], [0, 0]):
    print(row)

[0, 1, -1, 7]
[1, 2, -1, 6]
[-1, 3, 4, 5]


`-1` nghĩa là không tới được (tường hoặc bị tường bao kín).

### 3.2 So sánh tuple theo thứ tự từ điển

`(2, 3, 1, 2) < (2, 3, 2, 0)` vì hai thành phần đầu bằng nhau, thành phần thứ ba `1 < 2`. Đây chính xác là luật xếp hạng của đề ⇒ khóa sắp xếp là `(dist, price, row, col)`.

In [3]:
items = [(2, 3, 1, 2), (2, 2, 2, 1), (3, 3, 1, 1), (4, 2, 0, 1)]
print(sorted(items))
print((2, 3, 1, 2) < (2, 3, 2, 0))

[(2, 2, 2, 1), (2, 3, 1, 2), (3, 3, 1, 1), (4, 2, 0, 1)]
True


### 3.3 `heapq` — lấy k phần tử nhỏ nhất

- `heapq.nsmallest(k, items)` trả về `k` phần tử nhỏ nhất đã sắp xếp, O(N log k).
- Tự duy trì **max-heap kích thước k** bằng cách đẩy khóa **âm** (heapq là min-heap): nếu heap đầy và phần tử mới tốt hơn phần tử tệ nhất trong heap thì thay.

In [4]:
data = [(5, "e"), (1, "a"), (4, "d"), (2, "b"), (3, "c")]
print(heapq.nsmallest(3, data))
h = []
for key, name in data:
    heapq.heappush(h, (-key, name))
    if len(h) > 3: heapq.heappop(h)         # bỏ phần tử LỚN nhất (khóa âm nhỏ nhất)
print(sorted((-a, b) for a, b in h))

[(1, 'a'), (2, 'b'), (3, 'c')]
[(1, 'a'), (2, 'b'), (3, 'c')]


### 3.4 BFS theo tầng

Xử lý hết các ô cùng khoảng cách `d` trong một vòng `for _ in range(len(q))`. Như vậy ta biết chính xác "tầng `d`" gồm những ô nào — hữu ích để **dừng sớm**: các tầng sau luôn có hạng thấp hơn mọi món ở tầng trước.

In [5]:
q = deque([(0, 0)]); seen = {(0, 0)}; g = examples[0][0]; d = 0
while q:
    layer = list(q)
    print(f"tầng {d}: {layer}")
    for _ in range(len(q)):
        r, c = q.popleft()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 4 and g[nr][nc] and (nr, nc) not in seen:
                seen.add((nr, nc)); q.append((nr, nc))
    d += 1

tầng 0: [(0, 0)]
tầng 1: [(1, 0), (0, 1)]
tầng 2: [(1, 1)]
tầng 3: [(2, 1)]
tầng 4: [(2, 2)]
tầng 5: [(2, 3)]
tầng 6: [(1, 3)]
tầng 7: [(0, 3)]


## 4. Ý tưởng nền tảng / trực giác

Bài gồm hai phần tách biệt:
1. **Khoảng cách**: tiêu chí số 1 là khoảng cách đường đi ngắn nhất (có tường), không phải khoảng cách Manhattan ⇒ cần BFS.
2. **Xếp hạng**: với mỗi món đạt giá, có khóa `(dist, price, r, c)`; lấy `k` khóa nhỏ nhất.

Tính tay ví dụ 2 (start `(2,3)`, giá `[2,3]`):

| ô | giá | khoảng cách | khóa |
|---|---|---|---|
| (2,1) | 2 | 2 | (2, 2, 2, 1) |
| (1,2) | 3 | 2 | (2, 3, 1, 2) |
| (1,1) | 3 | 3 | (3, 3, 1, 1) |
| (0,1) | 2 | 4 | (4, 2, 0, 1) |
| (2,2) | 5 | 1 | giá ngoài khoảng — bỏ |

Sắp xếp khóa ⇒ `(2,1)`, `(1,2)`, … ⇒ `k = 2` cho `[[2,1],[1,2]]`. ✔

Chú ý ô `(2,2)` giá 5 vẫn **đi qua được** (chỉ không được tính là món quan tâm).

In [6]:
g, p, s, k, _ = examples[1]
dist = bfs_dist(g, s)
cands = [(dist[r][c], g[r][c], r, c) for r in range(len(g)) for c in range(len(g[0]))
         if dist[r][c] >= 0 and p[0] <= g[r][c] <= p[1]]
for key in sorted(cands):
    print("khóa", key)

khóa (2, 2, 2, 1)
khóa (2, 3, 1, 2)
khóa (3, 3, 1, 1)
khóa (4, 2, 0, 1)


## 5. Cách 1 — Brute Force / baseline: BFS + sắp xếp toàn bộ

**Ý tưởng:** BFS một lần để có `dist` của mọi ô; gom mọi món đạt giá và tới được; sắp xếp theo khóa; lấy `k` phần tử đầu.

In [7]:
def bfs_sort_all(grid, pricing, start, k):
    low, high = pricing
    dist = bfs_dist(grid, start)
    cands = []
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            if dist[r][c] >= 0 and low <= v <= high:
                cands.append((dist[r][c], v, r, c))
    cands.sort()
    return [[r, c] for _, _, r, c in cands[:k]]

for g, p, s, k, exp in examples:
    got = bfs_sort_all(g, p, s, k)
    print(got, "OK" if got == exp else "SAI")

[[0, 1], [1, 1], [2, 1]] OK
[[2, 1], [1, 2]] OK
[[2, 1], [2, 0]] OK


**Độ phức tạp:** BFS O(m·n); sắp xếp O(N log N) với N = số món đạt giá (≤ m·n). Tổng **O(mn log mn)**, bộ nhớ O(mn).

Thực ra đây đã là lời giải chấp nhận được (m·n ≤ 10⁵). "Nút thắt" nhẹ: ta sắp xếp **mọi** món dù chỉ cần `k` món, và BFS **toàn bộ** lưới dù `k` món đầu có thể nằm rất gần `start`.

### 5.1 Baseline "rất thô": tính khoảng cách bằng relax lặp (để kiểm chứng)

Để có một bộ kiểm chứng **độc lập** với BFS, tính khoảng cách bằng cách lặp: `dist[v] = min(dist[v], dist[u] + 1)` cho mọi cạnh, đến khi không đổi (giống Bellman–Ford). Chậm (O((mn)²)) nhưng hiển nhiên đúng.

In [8]:
def relax_dist(grid, start):
    m, n = len(grid), len(grid[0]); INF = float("inf")
    dist = [[INF] * n for _ in range(m)]; dist[start[0]][start[1]] = 0
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 0 or dist[r][c] == INF: continue
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and dist[r][c] + 1 < dist[nr][nc]:
                        dist[nr][nc] = dist[r][c] + 1; changed = True
    return dist

def brute(grid, pricing, start, k):
    dist = relax_dist(grid, start)
    cands = [(dist[r][c], grid[r][c], r, c) for r in range(len(grid)) for c in range(len(grid[0]))
             if dist[r][c] != float("inf") and pricing[0] <= grid[r][c] <= pricing[1]]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

for g, p, s, k, exp in examples:
    print(brute(g, p, s, k), "OK" if brute(g, p, s, k) == exp else "SAI")

[[0, 1], [1, 1], [2, 1]] OK
[[2, 1], [1, 2]] OK
[[2, 1], [2, 0]] OK


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — BFS là đủ cho tiêu chí 1:** mọi bước trọng số 1 ⇒ thứ tự BFS lấy ô ra khỏi hàng đợi là **không giảm theo khoảng cách**, và lần chạm đầu tiên là khoảng cách ngắn nhất.

**Insight 2 — tầng quyết định trước:** mọi món ở tầng `d` có hạng **cao hơn** mọi món ở tầng `d + 1` (tiêu chí khoảng cách đứng đầu). Vậy:
- **Nếu** sau khi gom hết tầng `d` ta đã có ≥ `k` món **thì** dừng BFS — các tầng sau không thể chen vào.
- Trong một tầng, sắp xếp theo `(price, row, col)` rồi lấy đủ phần còn thiếu.

**Insight 3 — chỉ cần k tốt nhất:** dùng `heapq.nsmallest(k, …)` (O(N log k)) hoặc max-heap kích thước `k` thay vì sắp xếp toàn bộ.

**Bất biến BFS theo tầng:** khi bắt đầu xử lý tầng `d`, hàng đợi chứa **đúng** các ô có khoảng cách `d` (chưa xử lý), và mọi ô khoảng cách `< d` đã được thăm và đánh giá.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS (gom khóa ngay trong BFS) + `heapq.nsmallest`

In [9]:
def bfs_nsmallest(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing
    r0, c0 = start
    seen = [[False] * n for _ in range(m)]; seen[r0][c0] = True
    q = deque([(r0, c0, 0)]); cands = []
    while q:
        r, c, d = q.popleft()
        if low <= grid[r][c] <= high:
            cands.append((d, grid[r][c], r, c))
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True; q.append((nr, nc, d + 1))
    return [[r, c] for _, _, r, c in heapq.nsmallest(k, cands)]

for g, p, s, k, exp in examples:
    got = bfs_nsmallest(g, p, s, k)
    print(got, "OK" if got == exp else "SAI")

[[0, 1], [1, 1], [2, 1]] OK
[[2, 1], [1, 2]] OK
[[2, 1], [2, 0]] OK


**Độ phức tạp cách 2:** O(mn + N log k), bộ nhớ O(mn).

### 7.2 Cách 3 — BFS theo tầng + dừng sớm

In [10]:
def bfs_layers_early(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing
    r0, c0 = start
    seen = [[False] * n for _ in range(m)]; seen[r0][c0] = True
    q = deque([(r0, c0)]); ans = []
    while q and len(ans) < k:
        layer = []
        for _ in range(len(q)):
            r, c = q.popleft()
            if low <= grid[r][c] <= high:
                layer.append((grid[r][c], r, c))
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                    seen[nr][nc] = True; q.append((nr, nc))
        layer.sort()
        for _, r, c in layer[:k - len(ans)]:
            ans.append([r, c])
    return ans

for g, p, s, k, exp in examples:
    got = bfs_layers_early(g, p, s, k)
    print(got, "OK" if got == exp else "SAI")

[[0, 1], [1, 1], [2, 1]] OK
[[2, 1], [1, 2]] OK
[[2, 1], [2, 0]] OK


**Độ phức tạp cách 3:** worst O(mn log mn) (khi phải duyệt hết), nhưng thực tế thường dừng rất sớm khi `k` nhỏ: chỉ duyệt các tầng tới khi đủ `k` món. Bộ nhớ O(mn) cho `seen`.

### 7.3 Cách 4 — BFS + max-heap kích thước k (bộ nhớ heap O(k))

Đẩy khóa "đảo dấu" để heap là max-heap; khi heap vượt `k` thì bỏ phần tử **tệ nhất**.

In [11]:
def bfs_maxheap(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing
    r0, c0 = start
    seen = [[False] * n for _ in range(m)]; seen[r0][c0] = True
    q = deque([(r0, c0, 0)]); h = []
    while q:
        r, c, d = q.popleft()
        v = grid[r][c]
        if low <= v <= high:
            heapq.heappush(h, (-d, -v, -r, -c))
            if len(h) > k:
                heapq.heappop(h)              # bỏ món hạng THẤP nhất
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True; q.append((nr, nc, d + 1))
    best = sorted((-a, -b, -x, -y) for a, b, x, y in h)
    return [[x, y] for _, _, x, y in best]

for g, p, s, k, exp in examples:
    got = bfs_maxheap(g, p, s, k)
    print(got, "OK" if got == exp else "SAI")

[[0, 1], [1, 1], [2, 1]] OK
[[2, 1], [1, 2]] OK
[[2, 1], [2, 0]] OK


Đảo dấu **tất cả** các thành phần làm thứ tự tuple đảo ngược hoàn toàn ⇒ phần tử nhỏ nhất của heap (đỉnh) chính là món hạng thấp nhất trong `k` món đang giữ.

**Độ phức tạp cách 4:** O(mn + N log k), heap O(k).

## 8. TRACE từng bước

Trace cách 3 (theo tầng + dừng sớm) trên ví dụ 1 với `k = 3`.

| tầng d | các ô ở tầng | món đạt giá [2,5] (giá, hàng, cột) | ans sau tầng |
|---|---|---|---|
| 0 | (0,0) | — (giá 1) | [] |
| 1 | (0,1), (1,0) | (2,0,1) | [[0,1]] |
| 2 | (1,1) | (3,1,1) | [[0,1],[1,1]] |
| 3 | (2,1) | (2,2,1) | [[0,1],[1,1],[2,1]] ⇒ đủ k, **dừng** |

Ô `(2,2)` (giá 5, tầng 4) không bao giờ được xử lý.

In [12]:
def trace_layers(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing; r0, c0 = start
    seen = {(r0, c0)}; q = deque([(r0, c0)]); ans = []; d = 0
    while q and len(ans) < k:
        cells = list(q); layer = []
        for _ in range(len(q)):
            r, c = q.popleft()
            if low <= grid[r][c] <= high: layer.append((grid[r][c], r, c))
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and (nr, nc) not in seen:
                    seen.add((nr, nc)); q.append((nr, nc))
        layer.sort()
        ans += [[r, c] for _, r, c in layer[:k - len(ans)]]
        print(f"tầng {d}: ô {cells}, món {layer}, ans = {ans}")
        d += 1
    print("dừng:", "đủ k" if len(ans) >= k else "hết ô"); return ans

trace_layers(*examples[0][:4])

tầng 0: ô [(0, 0)], món [], ans = []
tầng 1: ô [(1, 0), (0, 1)], món [(2, 0, 1)], ans = [[0, 1]]
tầng 2: ô [(1, 1)], món [(3, 1, 1)], ans = [[0, 1], [1, 1]]
tầng 3: ô [(2, 1)], món [(2, 2, 1)], ans = [[0, 1], [1, 1], [2, 1]]
dừng: đủ k


[[0, 1], [1, 1], [2, 1]]

Trace ví dụ 2 — hai món cùng tầng 2, xếp theo giá:

In [13]:
trace_layers(*examples[1][:4])

tầng 0: ô [(2, 3)], món [], ans = []
tầng 1: ô [(1, 3), (2, 2)], món [], ans = []
tầng 2: ô [(0, 3), (1, 2), (2, 1)], món [(2, 2, 1), (3, 1, 2)], ans = [[2, 1], [1, 2]]
dừng: đủ k


[[2, 1], [1, 2]]

## 9. Code nộp LeetCode

Chọn cách 3 (BFS theo tầng + dừng sớm). Không sửa `grid` của người gọi.

In [14]:
from typing import List
from collections import deque

class Solution:
    def highestRankedKItems(self, grid: List[List[int]], pricing: List[int], start: List[int], k: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        low, high = pricing
        r0, c0 = start
        seen = [[False] * n for _ in range(m)]
        seen[r0][c0] = True
        q = deque([(r0, c0)])
        ans = []
        while q and len(ans) < k:
            layer = []                               # các món trong tầng hiện tại
            for _ in range(len(q)):
                r, c = q.popleft()
                if low <= grid[r][c] <= high:
                    layer.append((grid[r][c], r, c))
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != 0 and not seen[nr][nc]:
                        seen[nr][nc] = True
                        q.append((nr, nc))
            layer.sort()                             # (giá, hàng, cột)
            for _, r, c in layer:
                if len(ans) == k:
                    break
                ans.append([r, c])
        return ans

In [15]:
sol = Solution()
for g, p, s, k, exp in examples:
    got = sol.highestRankedKItems(g, p, s, k)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[[0, 1], [1, 1], [2, 1]] | kỳ vọng [[0, 1], [1, 1], [2, 1]] | OK
[[2, 1], [1, 2]] | kỳ vọng [[2, 1], [1, 2]] | OK
[[2, 1], [2, 0]] | kỳ vọng [[2, 1], [2, 0]] | OK


## 10. Vì sao thuật toán đúng

**Khoảng cách đúng:** BFS với trọng số 1: khi một ô được đẩy vào hàng đợi lần đầu từ ô ở tầng `d`, nó có khoảng cách `d + 1`, và không có đường ngắn hơn (nếu có thì nó đã được chạm tới ở tầng `≤ d`). Đánh dấu `seen` khi đẩy đảm bảo mỗi ô ở đúng một tầng.

**Thứ tự hạng đúng:** khóa `(dist, price, row, col)` trùng đúng luật hạng; các món được thêm theo **tầng tăng dần**, và trong tầng sắp theo `(price, row, col)` ⇒ dãy `ans` là dãy tăng dần theo khóa đầy đủ.

**Dừng sớm đúng:** khi `ans` đủ `k` món sau tầng `d`, mọi món chưa xét có `dist > d` ⇒ hạng thấp hơn mọi món trong `ans` ⇒ không thay đổi `k` món tốt nhất. Nếu tầng `d` có nhiều hơn số còn thiếu, ta lấy đúng những món nhỏ nhất theo `(price, row, col)` — vì cùng `dist` nên đó là những món hạng cao nhất. ∎

**Ô xuất phát:** `grid[start] > 0` theo đề; nếu giá của nó trong khoảng thì nó là món duy nhất ở tầng 0 với hạng cao nhất — thuật toán xét nó ngay vòng đầu.

## 11. Phân tích Time / Space Complexity

`V = m·n ≤ 10⁵`, `N` = số món đạt giá và tới được.

| Cách | Time | Space |
|---|---|---|
| 1. BFS + sort toàn bộ | O(V + N log N) | O(V) |
| 1b. Relax lặp (kiểm chứng) | O(V²) | O(V) |
| 2. BFS + nsmallest | O(V + N log k) | O(V) |
| 3. Tầng + dừng sớm | O(V log V) worst, thường nhỏ hơn nhiều | O(V) |
| 4. BFS + max-heap k | O(V + N log k) | O(V) + heap O(k) |

In [16]:
rng = random.Random(2146)
m, n = 300, 333
big = [[0 if rng.random() < 0.2 else rng.randint(1, 100) for _ in range(n)] for _ in range(m)]
big[150][150] = 1
for kk in [5, 50000]:
    res = []
    for name, f in [("bfs_sort_all", bfs_sort_all), ("bfs_nsmallest", bfs_nsmallest),
                    ("bfs_layers_early", bfs_layers_early), ("bfs_maxheap", bfs_maxheap), ("Solution", Solution().highestRankedKItems)]:
        t = time.perf_counter(); r = f(big, [10, 60], [150, 150], kk); dt = time.perf_counter() - t
        res.append(r); print(f"k={kk:<6} {name:<17} {dt*1000:6.0f} ms")
    assert all(r == res[0] for r in res)

k=5      bfs_sort_all          85 ms


k=5      bfs_nsmallest         52 ms
k=5      bfs_layers_early       0 ms


k=5      bfs_maxheap           50 ms
k=5      Solution               0 ms
k=50000  bfs_sort_all          70 ms


k=50000  bfs_nsmallest         88 ms


k=50000  bfs_layers_early      57 ms
k=50000  bfs_maxheap           84 ms


k=50000  Solution             101 ms


Với `k = 5`, cách dừng sớm nhanh hơn hẳn vì chỉ duyệt vài tầng quanh `start`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "bfs_sort_all": bfs_sort_all, "bfs_nsmallest": bfs_nsmallest,
       "bfs_layers_early": bfs_layers_early, "bfs_maxheap": bfs_maxheap, "Solution": Solution().highestRankedKItems}
edge = [e for e in examples] + [
    ([[5]], [2, 5], [0, 0], 1, [[0, 0]]),                    # ô xuất phát là món đạt giá
    ([[1]], [2, 5], [0, 0], 1, []),                          # không có món
    ([[3, 0, 3]], [2, 5], [0, 0], 2, [[0, 0]]),             # món bên kia tường: không tới được
    ([[2, 2], [2, 2]], [2, 2], [0, 0], 4, [[0, 0], [0, 1], [1, 0], [1, 1]]),   # hòa: theo hàng rồi cột
    ([[1, 4, 3], [2, 1, 1]], [2, 4], [1, 1], 3, [[1, 0], [0, 1], [0, 2]]),     # tầng 1: giá 2 trước giá 4
]
for g, p, s, k, exp in edge:
    for name, f in ALL.items():
        gg = copy.deepcopy(g)
        got = f(gg, p, s, k)
        assert gg == g, (name, "đã sửa grid")
        assert got == exp, (name, g, got, exp)
print("Edge case OK:", len(edge))
for g, p, s, k, exp in edge:
    print(f"{g} {p} {s} k={k} -> {Solution().highestRankedKItems(g, p, s, k)}")

Edge case OK: 8
[[1, 2, 0, 1], [1, 3, 0, 1], [0, 2, 5, 1]] [2, 5] [0, 0] k=3 -> [[0, 1], [1, 1], [2, 1]]
[[1, 2, 0, 1], [1, 3, 3, 1], [0, 2, 5, 1]] [2, 3] [2, 3] k=2 -> [[2, 1], [1, 2]]
[[1, 1, 1], [0, 0, 1], [2, 3, 4]] [2, 3] [0, 0] k=3 -> [[2, 1], [2, 0]]
[[5]] [2, 5] [0, 0] k=1 -> [[0, 0]]
[[1]] [2, 5] [0, 0] k=1 -> []
[[3, 0, 3]] [2, 5] [0, 0] k=2 -> [[0, 0]]
[[2, 2], [2, 2]] [2, 2] [0, 0] k=4 -> [[0, 0], [0, 1], [1, 0], [1, 1]]
[[1, 4, 3], [2, 1, 1]] [2, 4] [1, 1] k=3 -> [[1, 0], [0, 1], [0, 2]]


Case cuối (tự dựng): start `(1,1)`; tầng 1 gồm `(0,1)` giá 4, `(1,0)` giá 2, `(1,2)` giá 1 (không đạt giá) ⇒ thứ tự trong tầng 1: `(1,0)` (giá 2) rồi `(0,1)` (giá 4); tầng 2: `(0,0)` giá 1 (bỏ), `(0,2)` giá 3 ⇒ đáp án `[[1,0],[0,1],[0,2]]`. Output của `Solution` ở trên phải khớp điều này — ta assert ở cell kế.

In [18]:
assert Solution().highestRankedKItems([[1, 4, 3], [2, 1, 1]], [2, 4], [1, 1], 3) == [[1, 0], [0, 1], [0, 2]]
print("case tự dựng khớp tính tay")

case tự dựng khớp tính tay


### 12.2 Đối chiếu ngẫu nhiên với brute force (relax lặp) — lưới nhỏ, seed cố định

In [19]:
rng = random.Random(42)
for _ in range(800):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[0 if rng.random() < 0.25 else rng.randint(1, 6) for _ in range(n)] for _ in range(m)]
    sr, sc = rng.randrange(m), rng.randrange(n)
    if g[sr][sc] == 0: g[sr][sc] = 1
    lo = rng.randint(2, 6); hi = rng.randint(lo, 6); k = rng.randint(1, m * n)
    ref = brute(g, [lo, hi], [sr, sc], k)
    for name, f in ALL.items():
        gg = copy.deepcopy(g)
        assert f(gg, [lo, hi], [sr, sc], k) == ref, (name, g, lo, hi, sr, sc, k)
        assert gg == g
print("800 case ngẫu nhiên OK")

800 case ngẫu nhiên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng khoảng cách Manhattan (bỏ qua tường)

In [20]:
def wrong_manhattan(grid, pricing, start, k):
    low, high = pricing; r0, c0 = start
    cands = [(abs(r - r0) + abs(c - c0), v, r, c) for r, row in enumerate(grid) for c, v in enumerate(row)
             if low <= v <= high]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

g, p, s, k, exp = examples[2]
print("bản sai :", wrong_manhattan(g, p, s, k))
print("bản đúng:", Solution().highestRankedKItems(g, p, s, k))

bản sai : [[2, 0], [2, 1]]
bản đúng: [[2, 1], [2, 0]]


**Vì sao sai:** tường buộc phải đi vòng: `(2,0)` có Manhattan 2 nhưng đường thật dài 6. Ngoài ra bản sai còn tính cả món **không tới được**. **Sửa:** BFS.

### Lỗi 2 — Sai thứ tự tiêu chí (giá trước khoảng cách)

In [21]:
def wrong_order(grid, pricing, start, k):
    dist = bfs_dist(grid, start)
    cands = [(v, dist[r][c], r, c) for r, row in enumerate(grid) for c, v in enumerate(row)
             if dist[r][c] >= 0 and pricing[0] <= v <= pricing[1]]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

g, p, s, k, exp = examples[0]
print("bản sai :", wrong_order(g, p, s, k))
print("bản đúng:", Solution().highestRankedKItems(g, p, s, k))

bản sai : [[0, 1], [2, 1], [1, 1]]
bản đúng: [[0, 1], [1, 1], [2, 1]]


**Sửa:** khóa phải là `(dist, price, row, col)` đúng thứ tự đề.

### Lỗi 3 — Quên xét ô xuất phát

In [22]:
def wrong_skip_start(grid, pricing, start, k):
    dist = bfs_dist(grid, start)
    cands = [(dist[r][c], v, r, c) for r, row in enumerate(grid) for c, v in enumerate(row)
             if dist[r][c] > 0 and pricing[0] <= v <= pricing[1]]     # SAI: > 0 bỏ mất start
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

g = [[3, 2], [1, 1]]
print("bản sai :", wrong_skip_start(g, [2, 3], [0, 0], 2))
print("bản đúng:", Solution().highestRankedKItems(g, [2, 3], [0, 0], 2))

bản sai : [[0, 1]]
bản đúng: [[0, 0], [0, 1]]


**Sửa:** ô xuất phát có khoảng cách 0 và có thể là một món hợp lệ — hạng cao nhất.

### Lỗi 4 — Đánh dấu `seen` khi **lấy ra** thay vì khi **đẩy vào** (đẩy trùng)

In [23]:
def wrong_mark_late(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing
    seen = set(); q = deque([(start[0], start[1], 0)]); cands = []; pushes = 0
    while q:
        r, c, d = q.popleft()
        if (r, c) in seen: continue
        seen.add((r, c))                                    # đánh dấu muộn
        if low <= grid[r][c] <= high: cands.append((d, grid[r][c], r, c))
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and (nr, nc) not in seen:
                q.append((nr, nc, d + 1)); pushes += 1
    return pushes, [[r, c] for _, _, r, c in sorted(cands)[:k]]

g = [[1] * 30 for _ in range(30)]
pushes, res = wrong_mark_late(g, [2, 5], [0, 0], 3)
print(f"bản đánh dấu muộn: {pushes} lần đẩy vào hàng đợi cho {30*30} ô (trùng lặp nhiều)")

bản đánh dấu muộn: 1740 lần đẩy vào hàng đợi cho 900 ô (trùng lặp nhiều)


Kết quả vẫn đúng (nhờ `continue`), nhưng số lần đẩy gần gấp đôi–gấp ba số ô — tốn thời gian/bộ nhớ, và nếu quên dòng `continue` thì các món bị **tính trùng**. Phản ví dụ khi quên `continue`:

In [24]:
def wrong_mark_late_no_skip(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing
    seen = set(); q = deque([(start[0], start[1], 0)]); cands = []
    while q:
        r, c, d = q.popleft()
        seen.add((r, c))
        if low <= grid[r][c] <= high: cands.append((d, grid[r][c], r, c))
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and (nr, nc) not in seen:
                q.append((nr, nc, d + 1))
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

g = [[1, 1], [1, 2]]
print("bản sai :", wrong_mark_late_no_skip(g, [2, 2], [0, 0], 2))
print("bản đúng:", Solution().highestRankedKItems(g, [2, 2], [0, 0], 2))

bản sai : [[1, 1], [1, 1]]
bản đúng: [[1, 1]]


**Sửa:** đánh dấu `seen` **ngay khi đẩy** vào hàng đợi.

### Lỗi 5 — Sửa trực tiếp `grid` (đặt ô đã thăm = 0) rồi làm hỏng input

In [25]:
def mutate_grid(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); low, high = pricing; r0, c0 = start
    q = deque([(r0, c0, 0)]); cands = []
    if low <= grid[r0][c0] <= high: cands.append((0, grid[r0][c0], r0, c0))
    grid[r0][c0] = 0                                      # đánh dấu bằng cách phá input
    while q:
        r, c, d = q.popleft()
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc]:
                if low <= grid[nr][nc] <= high: cands.append((d + 1, grid[nr][nc], nr, nc))
                grid[nr][nc] = 0; q.append((nr, nc, d + 1))
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

g = copy.deepcopy(examples[0][0])
print("kết quả:", mutate_grid(g, [2, 5], [0, 0], 3), "| grid sau khi gọi:", g)
print("gọi lần 2 trên cùng grid:", mutate_grid(g, [2, 5], [0, 0], 3))

kết quả: [[0, 1], [1, 1], [2, 1]] | grid sau khi gọi: [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
gọi lần 2 trên cùng grid: []


**Vì sao "sai":** cách doocs (đặt `grid[r][c] = 0`) đúng trên LeetCode nhưng **phá hủy** input — gọi lại lần hai cho kết quả rỗng. **Sửa:** dùng mảng `seen` riêng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khoảng cách ngắn nhất trên lưới, mỗi bước 1, có tường" ⇒ **BFS**, không phải Manhattan.
- "Xếp hạng theo nhiều tiêu chí" ⇒ **tuple khóa** đúng thứ tự tiêu chí; tiêu chí "lớn hơn tốt hơn" thì đổi dấu.
- "Chỉ cần k tốt nhất" ⇒ `heapq.nsmallest` hoặc heap kích thước k; nếu tiêu chí đầu là khoảng cách BFS ⇒ **BFS theo tầng + dừng sớm**.
- Đánh dấu thăm **khi đẩy**.
- Bài cùng họ: 1091 Shortest Path in Binary Matrix, 994 Rotting Oranges, 1926 Nearest Exit from Entrance in Maze, 973 K Closest Points to Origin (top-k), 2512.

## 15. Biến thể của bài

### 15.1 Biến thể A — Ưu tiên **giá cao hơn** (thay vì rẻ hơn)

Chỉ cần đổi dấu giá trong khóa: `(dist, -price, row, col)`.

In [26]:
def highest_price_first(grid, pricing, start, k):
    dist = bfs_dist(grid, start)
    cands = [(dist[r][c], -v, r, c) for r, row in enumerate(grid) for c, v in enumerate(row)
             if dist[r][c] >= 0 and pricing[0] <= v <= pricing[1]]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

print(highest_price_first(*examples[1][:4]), "(so với bài gốc:", Solution().highestRankedKItems(*examples[1][:4]), ")")

[[1, 2], [2, 1]] (so với bài gốc: [[2, 1], [1, 2]] )


### 15.2 Biến thể B — Ô có **chi phí đi vào** khác nhau (trọng số) ⇒ Dijkstra

Nếu bước vào ô `(r, c)` tốn `w(r, c)` thay vì 1, BFS không còn đúng; dùng Dijkstra với `heapq`. Ta kiểm chứng: khi mọi trọng số = 1, Dijkstra cho cùng kết quả với BFS.

In [27]:
def dijkstra_rank(grid, cost, pricing, start, k):
    m, n = len(grid), len(grid[0]); INF = float("inf")
    dist = [[INF] * n for _ in range(m)]; r0, c0 = start; dist[r0][c0] = 0
    pq = [(0, r0, c0)]
    while pq:
        d, r, c = heapq.heappop(pq)
        if d > dist[r][c]: continue
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and d + cost[nr][nc] < dist[nr][nc]:
                dist[nr][nc] = d + cost[nr][nc]; heapq.heappush(pq, (dist[nr][nc], nr, nc))
    cands = [(dist[r][c], grid[r][c], r, c) for r in range(m) for c in range(n)
             if dist[r][c] < INF and pricing[0] <= grid[r][c] <= pricing[1]]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

for g, p, s, k, exp in examples:
    ones = [[1] * len(g[0]) for _ in g]
    assert dijkstra_rank(g, ones, p, s, k) == exp
g = examples[0][0]
heavy = [[1, 1, 1, 1], [1, 9, 1, 1], [1, 1, 1, 1]]           # đi vào (1,1) tốn 9
print("trọng số khác nhau:", dijkstra_rank(g, heavy, [2, 5], [0, 0], 4))

trọng số khác nhau: [[0, 1], [1, 1], [2, 1], [2, 2]]


### 15.3 Biến thể C — Nhiều điểm xuất phát (multi-source BFS)

Khoảng cách = khoảng cách tới điểm xuất phát **gần nhất**: khởi tạo hàng đợi với **mọi** điểm xuất phát (cùng `dist = 0`).

In [28]:
def multi_source_rank(grid, pricing, starts, k):
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]; q = deque()
    for r, c in starts:
        dist[r][c] = 0; q.append((r, c))
    while q:
        r, c = q.popleft()
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1; q.append((nr, nc))
    cands = [(dist[r][c], grid[r][c], r, c) for r in range(m) for c in range(n)
             if dist[r][c] >= 0 and pricing[0] <= grid[r][c] <= pricing[1]]
    return [[r, c] for _, _, r, c in sorted(cands)[:k]]

print(multi_source_rank(examples[0][0], [2, 5], [[0, 0], [2, 3]], 4))

[[0, 1], [2, 2], [2, 1], [1, 1]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS + sort | tính hết rồi sắp | O(V + N log N) | O(V) | dễ viết, dễ đúng |
| 1b. Relax lặp | khoảng cách kiểu Bellman–Ford | O(V²) | O(V) | kiểm chứng |
| 2. BFS + nsmallest | top-k bằng heap | O(V + N log k) | O(V) | k nhỏ, code ngắn |
| 3. Tầng + dừng sớm | tầng trước luôn hạng cao hơn | thường ≪ V | O(V) | **mặc định**, k nhỏ & lưới lớn |
| 4. BFS + max-heap k | giữ k tốt nhất | O(V + N log k) | O(V) + O(k) | streaming |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `grid = [[1,2],[3,4]]`, `pricing = [2,4]`, `start = [0,0]`, `k = 3`?

**Bài 2 (tính tay):** Với ví dụ 3, ghi khoảng cách BFS của mọi ô (−1 cho tường).

**Bài 3 (điền TODO):** Viết `bt3_todo(grid, pricing, start, k)` trả về kết quả **kèm khoảng cách**: list `[r, c, dist]`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(...)` cài cách 3 (theo tầng + dừng sớm) không nhìn code.

**Bài 5 (mini challenge):** Có thể **phá tối đa một bức tường** (biến một ô 0 thành 1) trước khi đi. Viết `bt5_todo(grid, pricing, start)` trả về **khoảng cách nhỏ nhất** tới một món đạt giá (−1 nếu không có). Gợi ý: BFS trên trạng thái `(r, c, đã phá chưa)`.

In [29]:
def bt3_todo(grid, pricing, start, k):
    # TODO: trả về [[r, c, dist], ...]
    pass

def bt4_todo(grid, pricing, start, k):
    # TODO: BFS theo tầng + dừng sớm
    pass

def bt5_todo(grid, pricing, start):
    # TODO: được phá tối đa một tường; khoảng cách nhỏ nhất tới món đạt giá
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: khoảng cách: (0,1) = 1, (1,0) = 1, (1,1) = 2. Mức 2: tầng 1: giá 2 tại (0,1), giá 3 tại (1,0). Mức 3: `[[0,1],[1,0],[1,1]]`.

**Bài 2.** Mức 1: hàng 1 có hai tường ở cột 0, 1. Mức 2: phải đi vòng qua cột 2. Mức 3: `[[0,1,2],[-1,-1,3],[6,5,4]]`.

**Bài 3.** Mức 1: tái dùng `bfs_dist`. Mức 2: sắp khóa `(dist, price, r, c)`. Mức 3: xuất `[r, c, dist]`.

**Bài 4.** Mức 1: vòng `for _ in range(len(q))` cho một tầng. Mức 2: gom món của tầng, sắp theo `(price, r, c)`. Mức 3: dừng khi `len(ans) == k`.

**Bài 5.** Mức 1: trạng thái thêm một bit "đã phá". Mức 2: đi vào ô 0 chỉ khi bit = 0, rồi bit thành 1. Mức 3: BFS, trả `d` lần đầu gặp ô có giá trong khoảng (lấy min qua cả hai trạng thái — BFS tự đảm bảo lần đầu là nhỏ nhất).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[[0,1],[1,0],[1,1]]`. **Bài 2:** `[[0,1,2],[-1,-1,3],[6,5,4]]`.

In [30]:
assert Solution().highestRankedKItems([[1, 2], [3, 4]], [2, 4], [0, 0], 3) == [[0, 1], [1, 0], [1, 1]]
assert bfs_dist(examples[2][0], [0, 0]) == [[0, 1, 2], [-1, -1, 3], [6, 5, 4]]

def bt3(grid, pricing, start, k):
    dist = bfs_dist(grid, start)
    cands = sorted((dist[r][c], v, r, c) for r, row in enumerate(grid) for c, v in enumerate(row)
                   if dist[r][c] >= 0 and pricing[0] <= v <= pricing[1])
    return [[r, c, d] for d, _, r, c in cands[:k]]

def bt4(grid, pricing, start, k):
    m, n = len(grid), len(grid[0]); lo, hi = pricing
    seen = {tuple(start)}; q = deque([tuple(start)]); ans = []
    while q and len(ans) < k:
        layer = []
        for _ in range(len(q)):
            r, c = q.popleft()
            if lo <= grid[r][c] <= hi: layer.append((grid[r][c], r, c))
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] and (nr, nc) not in seen:
                    seen.add((nr, nc)); q.append((nr, nc))
        for _, r, c in sorted(layer):
            if len(ans) < k: ans.append([r, c])
    return ans

def bt5(grid, pricing, start):
    m, n = len(grid), len(grid[0]); lo, hi = pricing
    s = (start[0], start[1], 0); seen = {s}; q = deque([(s, 0)])
    while q:
        (r, c, used), d = q.popleft()
        if lo <= grid[r][c] <= hi: return d
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n:
                nu = used + (grid[nr][nc] == 0)
                if nu <= 1 and (nr, nc, nu) not in seen:
                    seen.add((nr, nc, nu)); q.append(((nr, nc, nu), d + 1))
    return -1

def bt5_brute(grid, pricing, start):
    best = -1; m, n = len(grid), len(grid[0])
    options = [None] + [(r, c) for r in range(m) for c in range(n) if grid[r][c] == 0]
    for w in options:
        g = [row[:] for row in grid]
        if w: g[w[0]][w[1]] = 1
        dist = bfs_dist(g, start)
        ds = [dist[r][c] for r in range(m) for c in range(n) if dist[r][c] >= 0 and pricing[0] <= g[r][c] <= pricing[1]]
        if ds and (best == -1 or min(ds) < best): best = min(ds)
    return best

assert bt3(*examples[0][:4]) == [[0, 1, 1], [1, 1, 2], [2, 1, 3]]
rng = random.Random(1946)
for _ in range(600):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = [[0 if rng.random() < 0.35 else rng.randint(1, 5) for _ in range(n)] for _ in range(m)]
    sr, sc = rng.randrange(m), rng.randrange(n)
    if g[sr][sc] == 0: g[sr][sc] = 1
    lo = rng.randint(2, 5); hi = rng.randint(lo, 5); k = rng.randint(1, m * n)
    ref = brute(g, [lo, hi], [sr, sc], k)
    assert bt4(g, [lo, hi], [sr, sc], k) == ref
    assert [x[:2] for x in bt3(g, [lo, hi], [sr, sc], k)] == ref
    assert bt5(g, [lo, hi], [sr, sc]) == bt5_brute(g, [lo, hi], [sr, sc])
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: món hàng **không** được nằm trên ô tường bị phá (ô 0 không có giá), nên kiểm tra `grid[r][c]` gốc là đúng. Brute force thử phá từng bức tường (hoặc không phá) rồi BFS — xác nhận lời giải trạng thái `(r, c, used)`.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
seen = [[False]*n for _ in range(m)]; seen[r0][c0] = True
q = deque([(r0, c0)]); ans = []
while q and len(ans) < k:
    layer = []
    for _ in range(len(q)):                       # một tầng = một khoảng cách
        r, c = q.popleft()
        if low <= grid[r][c] <= high: layer.append((grid[r][c], r, c))
        for nr, nc in ((r+1,c),(r-1,c),(r,c+1),(r,c-1)):
            if 0<=nr<m and 0<=nc<n and grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True; q.append((nr, nc))
    for _, r, c in sorted(layer)[:k-len(ans)]: ans.append([r, c])
```

**Checklist:**
- [ ] Khoảng cách bằng **BFS** (có tường), không phải Manhattan.
- [ ] Khóa `(dist, price, row, col)` đúng thứ tự.
- [ ] Xét cả **ô xuất phát** (dist 0).
- [ ] Đánh dấu `seen` khi đẩy; không sửa `grid` nếu không cần.
- [ ] Ít hơn `k` món ⇒ trả tất cả.